# Экспериментальная валидация MA-VCG-QMIX

Этот notebook воспроизводит всю экспериментальную часть:

1. задаёт сценарии и сравниваемые методы;
2. запускает benchmark;
3. строит графики и LaTeX-таблицы;
4. подготавливает артефакты для главы диссертации.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import Image, Markdown, display

from src.learning.benchmark import BenchmarkRunner
from visualization.plot_results import ResultsVisualizer

os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/xdg-cache')

RESULTS_ROOT = Path('experiments/results/validation')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
runner = BenchmarkRunner(RESULTS_ROOT)
visualizer = ResultsVisualizer(RESULTS_ROOT)

## Конфигурация benchmark

In [ ]:
scenarios = runner.default_scenarios()
methods = runner.default_methods()

scenario_df = pd.DataFrame([
    {
        'scenario': scenario.name,
        'label': scenario.label,
        'episodes': scenario.training_config.num_episodes,
        'steps': scenario.training_config.max_steps_per_episode,
        'arrival_rate': scenario.env_config.task_lambda_arrival,
        'description': scenario.description,
    }
    for scenario in scenarios
])
method_df = pd.DataFrame([
    {
        'method': method.name,
        'label': method.label,
        'learning_enabled': method.learning_enabled,
        'description': method.description,
    }
    for method in methods
])

display(Markdown('### Сценарии'))
display(scenario_df)
display(Markdown('### Сравниваемые методы'))
display(method_df)

## Запуск симуляций

In [ ]:
summary = runner.run_suite(scenarios=scenarios, methods=methods, seed=42)
summary = summary.sort_values(['scenario', 'method']).reset_index(drop=True)
summary

## Генерация chapter artifacts

In [ ]:
visualizer.build_all()
display(Markdown('Сводные таблицы сохранены в `experiments/results/validation/tables/`, графики — в `experiments/results/validation/plots/`.'))

## Ключевые таблицы

In [ ]:
display(pd.read_csv(RESULTS_ROOT / 'tables' / 'summary_table.csv'))
display(pd.read_csv(RESULTS_ROOT / 'tables' / 'scenario_winners.csv'))

## Основные рисунки

In [ ]:
for figure_name in [
    'method_overview.png',
    'learning_curves.png',
    'baseline_dynamics.png',
    'load_spikes_dynamics.png',
    'heterogeneous_dynamics.png',
    'failures_dynamics.png',
    'fairness_welfare_scatter.png',
]:
    display(Markdown(f'### {figure_name}'))
    display(Image(filename=str(RESULTS_ROOT / 'plots' / figure_name)))

## Быстрые выводы для главы

In [ ]:
best_by_sw = (
    summary.sort_values('mean_social_welfare', ascending=False)
    .groupby('scenario_label', as_index=False)
    .first()[['scenario_label', 'method_label', 'mean_social_welfare']]
)
best_by_fairness = (
    summary.sort_values('mean_gini_payment', ascending=True)
    .groupby('scenario_label', as_index=False)
    .first()[['scenario_label', 'method_label', 'mean_gini_payment']]
)

display(Markdown('### Лучший метод по социальному благосостоянию'))
display(best_by_sw)
display(Markdown('### Лучший метод по равномерности платежей (минимум Gini)'))
display(best_by_fairness)